In [1]:

import os

from langchain_openai import ChatOpenAI

from utils.base64_util import audio_tobase64

model = ChatOpenAI(
    model='Qwen/Qwen3-Omni-30B-A3B-Instruct',
    base_url=os.environ.get('OPENAI_BASE_URL'),
    api_key=os.environ.get('OPENAI_API_KEY')
)

D:\Work\Langchain-learn-demo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 基于内存的聊天历史

In [2]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    # 系统提示
    ('system', '你是一个乐于助人的助手。尽你所能回答所有的问题。提供的聊天历史包含与你对话用户的相关信息'),
    # 聊天历史
    MessagesPlaceholder(variable_name='chat_history', optional=True),
    # 用户输入
    ('human', '{input}')
])

In [3]:
from langchain_core.chat_history import InMemoryChatMessageHistory

chain = prompt | model

store = {}


def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [4]:
from langchain_core.runnables import RunnableWithMessageHistory

# 创建带有聊天历史的chain
chain = RunnableWithMessageHistory(
    prompt | model,
    get_session_history,
    input_messages_key='input',
    history_messages_key='chat_history'
)

In [5]:
config = {
    "configurable": {
        "session_id": "user123"
    }
}
chain.invoke({'input': '你好'}, config=config)

AIMessage(content='你好！有什么我可以帮助你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 39, 'total_tokens': 47, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-Omni-30B-A3B-Instruct', 'system_fingerprint': '', 'id': '019da34a7fed9018d530ecc63933ba2f', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da34a-7f09-7a61-967e-c9f0c2688a39-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 39, 'output_tokens': 8, 'total_tokens': 47, 'input_token_details': {}, 'output_token_details': {}})

In [6]:
chain.invoke({'input': '你是谁'}, config=config)

AIMessage(content='我是阿里巴巴集团旗下的通义实验室自主研发的多模态超大规模语言模型，我叫通义千问Omni。我能够实时理解并融合文本、图像、音频、视频等多种模态信息，并支持用户输入图片、视频以及实现麦克风录音和摄像头录像。我不仅能够回答问题、创作文字，还能进行逻辑推理、编程等，甚至表达观点和玩游戏。有什么我可以帮你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 88, 'prompt_tokens': 59, 'total_tokens': 147, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-Omni-30B-A3B-Instruct', 'system_fingerprint': '', 'id': '019da34a819117d7553d523bd6ec7ba2', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da34a-8137-7962-aeef-d231a01bedb4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 59, 'output_tokens': 88, 'total_tokens': 147, 'input_token_details': {}, 'output_token_details': {}})

In [7]:
chain.invoke({'input': '我刚才说的第一句话是什么'}, config=config)

AIMessage(content='你刚才说的第一句话是：“你好”。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 163, 'total_tokens': 172, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-Omni-30B-A3B-Instruct', 'system_fingerprint': '', 'id': '019da34a866dc6f16fdece71ca98c771', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da34a-8634-7212-b12a-e35a438b30af-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 163, 'output_tokens': 9, 'total_tokens': 172, 'input_token_details': {}, 'output_token_details': {}})

In [10]:
for message in store['user123'].messages:
    print(type(message), message.content)

<class 'langchain_core.messages.human.HumanMessage'> 你好
<class 'langchain_core.messages.ai.AIMessage'> 你好！有什么我可以帮助你的吗？
<class 'langchain_core.messages.human.HumanMessage'> 你是谁
<class 'langchain_core.messages.ai.AIMessage'> 我是阿里巴巴集团旗下的通义实验室自主研发的多模态超大规模语言模型，我叫通义千问Omni。我能够实时理解并融合文本、图像、音频、视频等多种模态信息，并支持用户输入图片、视频以及实现麦克风录音和摄像头录像。我不仅能够回答问题、创作文字，还能进行逻辑推理、编程等，甚至表达观点和玩游戏。有什么我可以帮你的吗？
<class 'langchain_core.messages.human.HumanMessage'> 我刚才说的第一句话是什么
<class 'langchain_core.messages.ai.AIMessage'> 你刚才说的第一句话是：“你好”。


# 使用外部存储保存聊天历史

In [11]:
from sqlalchemy.ext.asyncio import create_async_engine
from langchain_community.chat_message_histories import SQLChatMessageHistory

# 使用异步调用
async_engine = create_async_engine("sqlite+aiosqlite:///data/chat_history.db")


def get_session_history_sql(session_id):
    return SQLChatMessageHistory(
        session_id=session_id,
        connection=async_engine
    )


# 创建带有聊天历史的chain
chain = RunnableWithMessageHistory(
    prompt | model,
    get_session_history_sql,
    input_messages_key='input',
    history_messages_key='chat_history'
)
await chain.ainvoke({'input': '你好'}, config=config)
await chain.ainvoke({'input': '你是谁'}, config=config)
await chain.ainvoke({'input': '我刚才说的第一句话是什么'}, config=config)

AIMessage(content='根据您在此前对话中的信息，您刚才说的第一句话是：“你好，我是蔡锷。”', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 435, 'total_tokens': 456, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-Omni-30B-A3B-Instruct', 'system_fingerprint': '', 'id': '019da34d0d71f2162a31981a7791eaf9', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da34d-0d26-7b83-af9b-4b8f5cb56af0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 435, 'output_tokens': 21, 'total_tokens': 456, 'input_token_details': {}, 'output_token_details': {}})

# 剪辑和摘要上下文

In [12]:

from langchain_core.messages import BaseMessage

summary_len = 5


async def summarize_context(current_input):
    session_id = current_input.get('config', {}).get("configurable", {}).get('session_id')
    if not session_id:
        raise ValueError('session_id is required')
    chat_history = get_session_history_sql(session_id)
    stored_messages = await chat_history.aget_messages()
    if len(stored_messages) <= summary_len:
        return False
    print("开始摘要历史对话")
    last_two_messages = stored_messages[-summary_len:]
    messages_to_be_summarized = stored_messages[:-summary_len]

    summarize_prompt = ChatPromptTemplate.from_messages([
        ('system', '请将下面消息进行剪辑和摘要，并返回结果'),
        MessagesPlaceholder(variable_name='chat_history'),
        ('human', '请生成包含上述对话核心内容的摘要，暴露重要的事实和决策'),
    ])

    summarize_chain = summarize_prompt | model
    summary_messages: BaseMessage = await summarize_chain.ainvoke({
        'chat_history': messages_to_be_summarized,
    })
    await chat_history.aclear()
    await chat_history.aadd_message(summary_messages)
    await chat_history.aadd_messages(last_two_messages)
    return True

In [13]:

from langchain_core.runnables import RunnablePassthrough

await get_session_history_sql(config['configurable']['session_id']).aclear()

# 创建带有聊天历史的chain
chain = (
        RunnablePassthrough.assign(messages_summarized=summarize_context) |
        RunnableWithMessageHistory(
            prompt | model,
            get_session_history_sql,
            input_messages_key='input',
            history_messages_key='chat_history'
        )
)
await chain.ainvoke({'input': '你好,今天天气很晴朗', 'config': config}, config=config)
await chain.ainvoke({'input': '你是谁', 'config': config}, config=config)
await chain.ainvoke({'input': '今天天气怎么样', 'config': config}, config=config)

AIMessage(content='我无法实时获取天气信息，但你可以通过天气预报应用或网站查看你所在地区的实时天气。不过既然你说今天天气很晴朗，那听起来是个适合外出活动的好日子呢！你有什么特别的安排吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 135, 'total_tokens': 183, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-Omni-30B-A3B-Instruct', 'system_fingerprint': '', 'id': '019da34d1271c46a1c0f86ecb90c0df0', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da34d-1232-7fa2-8e37-58a2e40cd117-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 135, 'output_tokens': 48, 'total_tokens': 183, 'input_token_details': {}, 'output_token_details': {}})

In [14]:
await get_session_history_sql(config['configurable']['session_id']).aget_messages()

[HumanMessage(content='你好,今天天气很晴朗', additional_kwargs={}, response_metadata={}),
 AIMessage(content='你好！听起来是个美好的日子呢。阳光明媚确实让人心情愉悦。您今天有什么特别的安排吗？希望您能好好享受这天气带来的好心情！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 45, 'total_tokens': 80, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-Omni-30B-A3B-Instruct', 'system_fingerprint': '', 'id': '019da34d0f0c06945c93f289aaccc813', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da34d-0ec7-7760-8a13-b57120db2649-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 45, 'output_tokens': 35, 'total_tokens': 80, 'input_token_details': {}, 'output_token_details': {}}),
 HumanMessage(content='你是谁', additional_kwargs={}, response_metadata={}),
 AIMessage(content='我是Qwen-Omni，是阿里巴巴集团旗下的通义实验室自主研发的多模态超大规模语言模型，有什么我可以帮助你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_u

- 上面的摘要思路是从历史聊天记录下手，导致历史记录丢失，下面进行优化

In [15]:
from langchain_core.messages import HumanMessage

summary_len = 5


async def summarize_context_v2(current_input):
    session_id = current_input.get('config', {}).get("configurable", {}).get('session_id')
    if not session_id:
        raise ValueError('session_id is required')
    chat_history = get_session_history_sql(session_id)
    stored_messages = await chat_history.aget_messages()
    if len(stored_messages) <= summary_len:
        return {"original_messages": stored_messages, "summary": None}
    print("开始摘要历史对话")
    last_two_messages = stored_messages[-summary_len:]
    messages_to_be_summarized = stored_messages[:-summary_len]

    summarize_prompt = ChatPromptTemplate.from_messages([
        ('system', '请将下面消息进行剪辑和摘要，并返回结果'),
        MessagesPlaceholder(variable_name='chat_history'),
        ('human', '请生成包含上述对话核心内容的摘要，暴露重要的事实和决策'),
    ])

    summarize_chain = summarize_prompt | model
    summary_message = await summarize_chain.ainvoke({
        'chat_history': messages_to_be_summarized,
    })
    return {
        "original_messages": last_two_messages,  # 保留的原始消息
        "summary": summary_message
    }


prompt = ChatPromptTemplate.from_messages([
    ('system', "你是一个乐于助人的助手。尽你所能回答所有问题。摘要：{summary}"),  # 动态注入系统消息
    MessagesPlaceholder(variable_name='chat_history', optional=True),
    ('human', '{input}')
])


async def invoke_and_save(context):
    session_id = context.get('config', {}).get("configurable", {}).get('session_id')
    if not session_id:
        raise ValueError('session_id is required')
    messages_summarized = context.get('messages_summarized', {})
    summary = messages_summarized.get('summary', '无摘要')
    chat_history = messages_summarized.get('original_messages')
    user_input = context.get('input')
    invoke_chain = prompt | model
    print(prompt.invoke({
        'input': user_input,
        'chat_history': chat_history,
        'summary': summary
    }))
    result = await invoke_chain.ainvoke({
        'input': user_input,
        'chat_history': chat_history,
        'summary': summary
    })
    chat_history_sql = get_session_history_sql(session_id)
    await chat_history_sql.aadd_message(HumanMessage(content=user_input))
    await chat_history_sql.aadd_message(result)
    return result



In [16]:
chain = (
        RunnablePassthrough.assign(messages_summarized=summarize_context_v2) |
        RunnablePassthrough.assign(
            input=lambda x: x['input'],
            system_message=lambda x: x['messages_summarized']['summary'] or '无摘要',
            chat_history=lambda x: x['messages_summarized']['original_messages']
        ) | invoke_and_save

)
await get_session_history_sql(config['configurable']['session_id']).aclear()
await chain.ainvoke({'input': '你好，我是蔡锷?', "config": {"configurable": {"session_id": "user123"}}},
                    config={"configurable": {"session_id": "user123"}})
await chain.ainvoke({'input': '我的名字叫什么?', "config": {"configurable": {"session_id": "user123"}}},
                    config={"configurable": {"session_id": "user123"}})
await chain.ainvoke({'input': '我喜欢打球!', "config": {"configurable": {"session_id": "user123"}}},
                    config={"configurable": {"session_id": "user123"}})
await chain.ainvoke(
    {'input': '用我的名字，写一篇短文，不超过200个字？', "config": {"configurable": {"session_id": "user123"}}},
    config={"configurable": {"session_id": "user123"}})

messages=[SystemMessage(content='你是一个乐于助人的助手。尽你所能回答所有问题。摘要：None', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，我是蔡锷?', additional_kwargs={}, response_metadata={})]
messages=[SystemMessage(content='你是一个乐于助人的助手。尽你所能回答所有问题。摘要：None', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，我是蔡锷?', additional_kwargs={}, response_metadata={}), AIMessage(content='你好，但在历史上，蔡锷是20世纪初中国的重要军事家和政治家，并非虚构人物。如果您以“蔡锷”之名提问，可能是想探讨历史人物或相关背景，欢迎您具体提问！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 37, 'total_tokens': 84, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-Omni-30B-A3B-Instruct', 'system_fingerprint': '', 'id': '019da34d155c382ae20f1a084416fb7f', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da34d-1500-77a2-9343-c1f4afae6563-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'out

AIMessage(content='蔡锷不仅是一位英勇的将领，也是一位热爱生活、积极进取的人。他常于闲暇之余练拳习武，也喜爱与将士们切磋球技。球台之间，他目光如炬，一挥拍动作流畅有力，展现将军的英姿与活力。在紧张的军旅生涯中，这项爱好让他保持敏锐的反应与强健的体魄。打球中见人格，运动里藏韬略，蔡锷用热爱诠释了“文武之道，一张一弛”的智慧。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 112, 'prompt_tokens': 462, 'total_tokens': 574, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-Omni-30B-A3B-Instruct', 'system_fingerprint': '', 'id': '019da34d1ded1ef4764bc5d0350c8285', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da34d-1d9d-7040-9591-e918a235bd2a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 462, 'output_tokens': 112, 'total_tokens': 574, 'input_token_details': {}, 'output_token_details': {}})

# 以下是错误案例 摘要后紧接着全量的历史聊天记录

In [20]:
from langchain_core.callbacks import BaseCallbackHandler


class DebugCallbackHandler(BaseCallbackHandler):
    """自定义调试回调"""

    def on_chat_model_start(self, serialized, messages, **kwargs):
        """在调用模型前触发"""
        print("\n" + "=" * 60)
        print("📤 发送到模型的完整消息:")
        print("=" * 60)
        for i, msg_list in enumerate(messages):
            print(f"\n消息批次 {i + 1}:")
            for j, msg in enumerate(msg_list):
                print(f"  [{j + 1}] 角色: {msg.type}")
                print(f"      内容: {msg.content[:200]}..." if len(
                    str(msg.content)) > 200 else f"      内容: {msg.content}")
        print("=" * 60 + "\n")

    def on_llm_end(self, response, **kwargs):
        """在模型返回后触发"""
        print("\n" + "=" * 60)
        print("📥 模型响应:")
        print("=" * 60)
        print(response.generations[0][0].text[:500])
        print("=" * 60 + "\n")


config = {
    "configurable": {
        "session_id": "user123"
    },
    "callbacks": [DebugCallbackHandler()]  # 添加这个回调
}
chain = (
        RunnablePassthrough.assign(messages_summarized=summarize_context_v2) |
        RunnablePassthrough.assign(
            input=lambda x: x['input'],
            summary=lambda x: x['messages_summarized']['summary'] or '无摘要',
            chat_history=lambda x: x['messages_summarized']['original_messages']
        ) |
        RunnableWithMessageHistory(
            prompt | model,
            get_session_history_sql,
            input_messages_key='input',
            history_messages_key='chat_history'
        )
)
await get_session_history_sql(config['configurable']['session_id']).aclear()
await chain.ainvoke({'input': '你好，我是蔡锷?', "config": {"configurable": {"session_id": "user123"}}},
                    config=config)
await chain.ainvoke({'input': '我的名字叫什么?', "config": {"configurable": {"session_id": "user123"}}},
                    config=config)
await chain.ainvoke({'input': '我喜欢打球!', "config": {"configurable": {"session_id": "user123"}}},
                    config=config)
await chain.ainvoke(
    {'input': '用我的名字，写一篇短文，不超过200个字？', "config": {"configurable": {"session_id": "user123"}}},
    config=config)


📤 发送到模型的完整消息:

消息批次 1:
  [1] 角色: system
      内容: 你是一个乐于助人的助手。尽你所能回答所有问题。摘要：无摘要
  [2] 角色: human
      内容: 你好，我是蔡锷?


📥 模型响应:
不，您不是蔡锷。蔡锷是中国近代著名的政治家、军事家，辛亥革命元勋之一。他于1882年出生，1916年逝世，享年仅34岁。蔡锷曾在云南发起护国战争，反对袁世凯称帝，为维护共和体制作出了重要贡献。如果您是自称是蔡锷，那可能是误解或虚构。如果是需要了解真实的蔡锷历史，我可以详细为您介绍。


📤 发送到模型的完整消息:

消息批次 1:
  [1] 角色: system
      内容: 你是一个乐于助人的助手。尽你所能回答所有问题。摘要：无摘要
  [2] 角色: human
      内容: 你好，我是蔡锷?
  [3] 角色: ai
      内容: 不，您不是蔡锷。蔡锷是中国近代著名的政治家、军事家，辛亥革命元勋之一。他于1882年出生，1916年逝世，享年仅34岁。蔡锷曾在云南发起护国战争，反对袁世凯称帝，为维护共和体制作出了重要贡献。如果您是自称是蔡锷，那可能是误解或虚构。如果是需要了解真实的蔡锷历史，我可以详细为您介绍。
  [4] 角色: human
      内容: 我的名字叫什么?


📥 模型响应:
您好，根据您提供的信息，您的名字是 **蔡锷**。

蔡锷（1882年－1916年），字松坡，是中国近代著名的军事将领、政治家，也是辛亥革命的重要人物之一。他在护国战争中发挥了重要作用，以反对袁世凯复辟帝制而闻名，被后人尊称为“护国军神”。


📤 发送到模型的完整消息:

消息批次 1:
  [1] 角色: system
      内容: 你是一个乐于助人的助手。尽你所能回答所有问题。摘要：无摘要
  [2] 角色: human
      内容: 你好，我是蔡锷?
  [3] 角色: ai
      内容: 不，您不是蔡锷。蔡锷是中国近代著名的政治家、军事家，辛亥革命元勋之一。他于1882年出生，1916年逝世，享年仅34岁。蔡锷曾在云南发起护国战争，反对袁世凯称帝，为维护共和体制作出了重要贡献。如果您是自称是蔡锷，那可能是误解或虚构。如果是需要了解真实的蔡锷历史，我可以详细为您介绍。
  

AIMessage(content='蔡锷，湘人也，自幼志向远大。偶好球胜于驰骋沙场。跑道上英姿勃发，球场中活力四射。马蹄声未绝，球影已动。矢志终生追求胜利之道，不止于军旅，亦在青年热血中闪光。真乃古今兼修，文武双全之士也！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 784, 'total_tokens': 865, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-Omni-30B-A3B-Instruct', 'system_fingerprint': '', 'id': '019da3512539f708a17aa46a29ea4e6c', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da351-24e9-7912-baa5-4e9c20db9b54-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 784, 'output_tokens': 81, 'total_tokens': 865, 'input_token_details': {}, 'output_token_details': {}})

# 音频转文字

In [23]:

from utils.base64_util import audio_to_base64

audio_base64 = audio_to_base64('data/audio.wav')
model.invoke([
    {
        "role": "user",
        "content": [
            {
                "type": "audio_url",
                "audio_url": {
                    "url": f"data:audio/wav;base64,{audio_base64}"
                }
            },
            {
                "type": "text",
                "text": "这段音频的内容是什么？"
            }
        ]
    }
])

AIMessage(content='Hello hello hello', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 3, 'prompt_tokens': 36, 'total_tokens': 39, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-Omni-30B-A3B-Instruct', 'system_fingerprint': '', 'id': '019da37fce98468d24ab4952430313cd', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019da37f-ccdb-7e01-a297-e3bde3c7ff2f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 3, 'total_tokens': 39, 'input_token_details': {}, 'output_token_details': {}})

In [25]:
from langchain_core.output_parsers import StrOutputParser

audio_base64 = audio_to_base64('data/audio.wav')
audio_to_text_chain_prompt = ChatPromptTemplate.from_messages([
    {
        "role": "user",
        "content": [
            {
                "type": "audio_url",
                "audio_url": {
                    "url": "data:audio/wav;base64,{audio_base64}"
                }
            },
            {
                "type": "text",
                "text": "这段音频的内容是什么？只输出音频内容文本"
            }
        ]
    }
])
audio_to_text_chain = audio_to_text_chain_prompt | model | StrOutputParser()

audio_to_text_chain.invoke({"audio_base64": audio_base64})

'Hello! Hello! Hello!'